# День 08. Упражнение 04
# Регрессия

## 0. Импорты

In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    train_test_split,
    cross_validate,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

## 1. Предобработка

1. Прочитай файл [`checker_regression.csv`](https://drive.google.com/file/d/1L8auBzJEghWFewznhhFpcrqxW4GqcfRY/view?usp=sharing).

In [43]:
df = pd.read_csv("../data/checker_regression.csv")
df.head()

,uid,num_commits,pageviews,AVG(diff)
0,user_1,62,28.0,NaN
1,user_1,62,28.0,NaN
2,user_1,62,28.0,NaN
3,user_1,62,28.0,NaN
4,user_1,62,28.0,NaN


2. Заполни пропущенные значения нулями (`0`).

In [44]:
df = df.fillna(0)
df.head()

,uid,num_commits,pageviews,AVG(diff)
0,user_1,62,28.0,0.0
1,user_1,62,28.0,0.0
2,user_1,62,28.0,0.0
3,user_1,62,28.0,0.0
4,user_1,62,28.0,0.0


In [45]:
df.value_counts().head(3)

uid      num_commits  pageviews  AVG(diff)
user_1   62           28.0       0.0          6
user_10  20           89.0       0.0          6
user_17  51           47.0       0.0          6
Name: count, dtype: int64

In [46]:
df = df.drop_duplicates().set_index("uid")
df.info()

<class 'pandas.DataFrame'>
Index: 29 entries, user_1 to user_8
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   num_commits  29 non-null     int64  
 1   pageviews    29 non-null     float64
 2   AVG(diff)    29 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 928.0+ bytes


3. Раздели набор данных на обучающую и тестовую выборки с параметрами `test_size=0.2`.

In [53]:
y = df["AVG(diff)"]
X = df.drop(columns=["AVG(diff)"])

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

## 2. Кросс-валидация

1. Напиши функцию `crossval`, которая принимает аргументы: `n_splits` для `KFold()`, `X`, `y`, модель (инициализированный класс с параметрами модели; не забудь о `random_state=21`) и возвращает для данного класса модели результат в таком виде:

```
train - 2696.4496895424836 | test - 1589.9979527104958
train - 2660.957874001452 | test - 2075.102636027137
train - 2847.315529246795 | test - 320.911928168403
train - 2500.7691099659237 | test - 4132.461382030178
train - 2643.927917295123 | test - 2237.8140952197878
train - 2396.295678819444 | test - 4509.650064742476
train - 2003.402267924976 | test - 8403.491474908551
train - 2531.876094212613 | test - 3135.944102735099
train - 2683.1795186023123 | test - 1796.01426292594
train - 2537.1192483996338 | test - 3439.29824116941
Average RMSE on crossval is 3164.0686140637476
```


In [81]:
def crossval(n_splits, X, y, model):
    """returns average neg_root_mean_squared_error (RMSE)"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
    # cross_val_score не подходит т.к сохр-т только метрику для внутритестового фолда каждой итерации а то, как модель отработала на на своей внутренней обучающей выборке, функция cross_val_score считает и далее стирает из памяти.
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=-1,
    )
    # Результаты работы функции cross_validate хранятся в обычном Python-словаре под ключами 'train_score' 'test_score'
    train_rmse = -cv_results["train_score"]
    test_rmse = -cv_results["test_score"]

    for train, test in zip(train_rmse, test_rmse):
        print(f"train - {train:<18} | test - {test}")

    mean_test_rmse = np.mean(test_rmse)
    print(f"Average RMSE on crossval is {mean_test_rmse}")

2. Запусти функцию для моделей `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`. Параметры можешь выбрать самостоятельно, но тщательно убедись, что они подходят.

In [82]:
crossval(10, X_train, y_train, LinearRegression())

train - 53.33351096491692  | test - 115.2394318363867
train - 62.409919772255556 | test - 69.35808794733022
train - 64.75146423244131  | test - 52.395419621714424
train - 64.04094981627755  | test - 78.18810821407222
train - 63.574089717062186 | test - 58.669554693832154
train - 65.40154560181439  | test - 32.10068854674122
train - 64.40600245557161  | test - 48.59322878192214
train - 61.215472663804725 | test - 82.41734194163652
train - 64.69372263304507  | test - 43.81163345544473
train - 62.80195740062014  | test - 70.47228900535454
Average RMSE on crossval is 65.1245784044435


## 3. Предсказания и оценка

1. Сделай предсказания на тестовом наборе данных для каждой из трёх моделей с финальными параметрами.
2. Построй график для каждой модели, где по оси x — фактическая средняя разница, а по оси y — предсказание модели.
3. Как должен выглядеть график в идеальном случае? Напиши ответ в markdown-ячейке в конце раздела.